# Clinical NLP Pipeline — Proof of Concept
**Cotiviti Intern Assessment | Clinical Natural Language Technology**

This notebook demonstrates a clinical NLP pipeline using the **Claude Fable API** to:
1. Extract structured clinical entities (Named Entity Recognition)
2. Summarize a clinical discharge note
3. Suggest ICD-10 codes with justification

**Stack:** Python · Anthropic SDK · Claude Fable 5

In [ ]:
# Install dependencies (run once)
# !pip install anthropic pandas

In [ ]:
import anthropic
import json
import os
from IPython.display import display, Markdown

# Initialize the Anthropic client
# Set your API key as an environment variable: export ANTHROPIC_API_KEY="sk-ant-..."
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

MODEL = "claude-fable-5"

print(f"Using model: {MODEL}")
print("Client initialized successfully.")

## 1. Sample Clinical Note

This is a **synthetic** clinical discharge note (no real patient data). In a production system, this input would come from an EHR via HL7/FHIR API or document ingestion pipeline.

In [ ]:
CLINICAL_NOTE = """
PATIENT: Jane Doe  |  DOB: 03/14/1962  |  MRN: 00123456
ADMISSION DATE: 07/28/2026  |  DISCHARGE DATE: 08/02/2026
ATTENDING PHYSICIAN: Dr. Michael Torres, MD, Internal Medicine

DISCHARGE SUMMARY

CHIEF COMPLAINT:
Patient presented with progressive shortness of breath, lower extremity edema, and fatigue
over the past two weeks.

HISTORY OF PRESENT ILLNESS:
Jane Doe is a 64-year-old female with a past medical history significant for type 2 diabetes
mellitus, hypertension, and chronic kidney disease stage 3a who presented to the emergency
department with worsening dyspnea on exertion and bilateral ankle edema. She reports a 10-pound
weight gain over the past 10 days. She denies chest pain, fever, or cough. Her most recent A1C
was 8.2% (6 months ago). She has been non-compliant with her low-sodium diet.

PHYSICAL EXAMINATION:
- Vitals: BP 158/94 mmHg, HR 88 bpm, RR 20/min, SpO2 93% on room air, Temp 98.6°F
- General: Alert and oriented, mildly distressed
- Cardiovascular: JVD present, S3 gallop auscultated
- Pulmonary: Bilateral basilar crackles
- Extremities: 2+ pitting edema bilateral lower extremities to the knees

DIAGNOSTIC RESULTS:
- BNP: 1,842 pg/mL (elevated)
- Troponin I: 0.02 ng/mL (within normal limits)
- Creatinine: 1.9 mg/dL (elevated from baseline 1.4)
- eGFR: 36 mL/min/1.73m² 
- Sodium: 131 mEq/L (hyponatremia)
- Echo (07/29/2026): EF 35%, dilated left ventricle, moderate mitral regurgitation
- CXR: Cardiomegaly, bilateral pleural effusions, pulmonary vascular congestion

DIAGNOSES:
1. Acute decompensated heart failure with reduced ejection fraction (HFrEF), EF 35%
2. Hypertensive heart disease
3. Type 2 diabetes mellitus, uncontrolled (A1C 8.2%)
4. Chronic kidney disease, stage 3a — acute-on-chronic exacerbation
5. Hyponatremia, likely dilutional
6. Moderate mitral regurgitation

HOSPITAL COURSE:
Patient was admitted and initiated on IV furosemide 80 mg BID with excellent diuresis (net
negative 3.2L over 5 days). She was transitioned to oral furosemide 40 mg daily on day 4.
Carvedilol 6.25 mg BID was initiated. Lisinopril was held during admission due to elevated
creatinine and restarted at 5 mg daily upon discharge after creatinine improvement to 1.6 mg/dL.
Endocrinology was consulted for diabetes management; metformin was held, and insulin glargine
10 units at bedtime was initiated. Nutrition counseling provided regarding fluid restriction and
low-sodium diet.

DISCHARGE MEDICATIONS:
1. Furosemide 40 mg oral daily
2. Carvedilol 6.25 mg oral twice daily
3. Lisinopril 5 mg oral daily
4. Insulin glargine 10 units subcutaneous at bedtime
5. Aspirin 81 mg oral daily
6. Atorvastatin 40 mg oral at bedtime

FOLLOW-UP:
- Cardiology clinic in 7 days (weight monitoring daily, call if +3 lbs in 24 hours)
- Primary care in 2 weeks
- Repeat BMP in 1 week to monitor renal function and electrolytes

ATTENDING PHYSICIAN SIGNATURE: Dr. Michael Torres, MD
"""

print("Clinical note loaded — length:", len(CLINICAL_NOTE), "characters")
print(CLINICAL_NOTE[:300], "...")

## 2. Clinical Named Entity Recognition (NER)

Extract structured entities from the clinical note using Claude Fable.

In [ ]:
def extract_clinical_entities(note: str) -> dict:
    """Extract structured clinical entities from a note using Claude Fable."""
    
    prompt = f"""You are a clinical NLP system. Extract structured information from the following 
clinical discharge note and return it as valid JSON only — no other text.

Extract these entity categories:
- patient: name, dob, mrn, age, sex
- admission: admit_date, discharge_date, attending_physician, los_days
- chief_complaint: (string)
- diagnoses: list of {{name, detail}}
- medications_at_discharge: list of {{name, dose, route, frequency}}
- lab_values: list of {{test, value, unit, flag}} where flag is "high", "low", or "normal"
- vital_signs: {{bp, hr, rr, spo2, temp}}
- procedures: list of procedures performed during admission
- follow_up: list of follow-up instructions

Clinical Note:
{note}

Return only valid JSON."""

    response = client.messages.create(
        model=MODEL,
        max_tokens=2048,
        messages=[{"role": "user", "content": prompt}]
    )
    
    raw = response.content[0].text.strip()
    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw)


print("Extracting clinical entities with Claude Fable...")
entities = extract_clinical_entities(CLINICAL_NOTE)
print("Done!")

In [ ]:
# Display results
display(Markdown("### Extracted Entities"))

display(Markdown(f"""
**Patient:** {entities.get('patient', {}).get('name')} | 
Age: {entities.get('patient', {}).get('age')} | 
Sex: {entities.get('patient', {}).get('sex')}

**Admission:** {entities.get('admission', {}).get('admit_date')} → {entities.get('admission', {}).get('discharge_date')} 
({entities.get('admission', {}).get('los_days')} days) | 
Attending: {entities.get('admission', {}).get('attending_physician')}

**Chief Complaint:** {entities.get('chief_complaint')}
"""))

display(Markdown("#### Diagnoses"))
for i, dx in enumerate(entities.get('diagnoses', []), 1):
    print(f"  {i}. {dx.get('name')} — {dx.get('detail', '')}")

display(Markdown("#### Discharge Medications"))
meds = entities.get('medications_at_discharge', [])
for med in meds:
    print(f"  • {med.get('name')} {med.get('dose')} {med.get('route')} {med.get('frequency')}")

display(Markdown("#### Lab Values"))
for lab in entities.get('lab_values', []):
    flag = lab.get('flag', 'normal').upper()
    symbol = "🔴" if flag == "HIGH" else ("🔵" if flag == "LOW" else "✅")
    print(f"  {symbol} {lab.get('test')}: {lab.get('value')} {lab.get('unit')} [{flag}]")

## 3. Clinical Summarization

Generate a concise, structured clinical summary suitable for a care transition handoff.

In [ ]:
def summarize_clinical_note(note: str) -> str:
    """Generate a concise clinical summary for care transition handoff."""
    
    prompt = f"""You are a clinical documentation specialist. Summarize the following discharge note 
into a concise, structured handoff summary for the outpatient care team. 

Use this format:
**Patient Overview:** [1-2 sentences: who, why admitted]
**Key Findings:** [3-4 bullet points of most critical clinical findings]
**Treatment:** [What was done and patient response]
**Discharge Regimen:** [Medications with key changes from prior]
**Critical Follow-Up Actions:** [What the outpatient team MUST do and when]
**Red Flag Signs:** [When patient should return to ED]

Be concise. Use clinical language appropriate for a physician audience.

Note:
{note}"""

    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text


print("Generating clinical summary with Claude Fable...")
summary = summarize_clinical_note(CLINICAL_NOTE)

display(Markdown("### Clinical Handoff Summary"))
display(Markdown(summary))

## 4. ICD-10 Code Suggestion

Suggest appropriate ICD-10-CM codes for each diagnosis with clinical justification — a direct application for Cotiviti's risk adjustment and coding workflows.

In [ ]:
def suggest_icd10_codes(note: str) -> list:
    """Suggest ICD-10-CM codes with justification from clinical note."""
    
    prompt = f"""You are a certified medical coder with expertise in ICD-10-CM. 
Review the clinical note and suggest the most appropriate ICD-10-CM codes for each documented condition.

Return valid JSON only — a list of objects with:
- code: ICD-10-CM code (e.g., "I50.22")
- description: official code description
- condition: condition from the note this maps to
- justification: brief clinical evidence from the note supporting this code
- hcc_relevant: true/false — whether this is a Hierarchical Condition Category code for risk adjustment
- principal_dx: true/false — whether this is the principal diagnosis

Note:
{note}

Return only valid JSON (list)."""

    response = client.messages.create(
        model=MODEL,
        max_tokens=2048,
        messages=[{"role": "user", "content": prompt}]
    )
    
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw)


print("Generating ICD-10 code suggestions with Claude Fable...")
icd_codes = suggest_icd10_codes(CLINICAL_NOTE)

display(Markdown("### ICD-10-CM Code Suggestions"))
display(Markdown("| Code | Description | HCC | Principal Dx |"))
display(Markdown("|------|-------------|-----|--------------|"))

for code in icd_codes:
    hcc = "✅ HCC" if code.get('hcc_relevant') else ""
    pdx = "⭐ Principal" if code.get('principal_dx') else ""
    print(f"**{code.get('code')}** | {code.get('description')} | {hcc} | {pdx}")
    print(f"  > Condition: {code.get('condition')}")
    print(f"  > Evidence: {code.get('justification')}")
    print()

## 5. Pipeline Summary

What this POC demonstrates about Clinical NLP capabilities:

In [ ]:
display(Markdown("""
## What This Pipeline Demonstrates

| Capability | Clinical NLP Task | Cotiviti Application |
|------------|-------------------|---------------------|
| Named Entity Recognition | Extract diagnoses, meds, labs, vitals | Risk adjustment, audit trail |
| Structured Output Generation | JSON from unstructured narrative | EHR integration, downstream analytics |
| Clinical Summarization | Handoff-ready summary | Care transitions, UR review |
| ICD-10 Code Suggestion | Diagnoses → billable codes with HCC flag | Risk adjustment, HCC capture, compliance |

### Key Design Principles for Production
- **Human-in-the-loop validation** before any code is finalized
- **De-identification** of PHI before sending to external APIs (or use private deployment)
- **Confidence scoring** to flag low-certainty extractions for review
- **Audit logging** of all model inputs and outputs for compliance

**Model used:** Claude Fable 5 (`claude-fable-5`) via Anthropic API  
**Total API calls:** 3 (NER, Summary, ICD coding)  
**Approx. latency per note:** 5–15 seconds  
"""))